# 1. The anatomy of an LLM variation operator

**What you get out of this notebook:** the four parts of the operator, running,
with the prompt, the sampled text, the verdict of the validator and the decision
of the search visible as four separate events.

Follows the section *Developing LLM variation operators* of the tutorial,
paragraph *Anatomy*, and Algorithm 1.

Nothing here calls a model over the network. The stand-in returns completions
from a fixed pool, which is what makes the run reproduce.

In [1]:
import _bootstrap  # puts the repository root on sys.path

from llm import MockLLM, load_pool
from search import build_and_validate

## The four parts

1. **Prompt.** What the operator is conditioned on.
2. **Sample.** One call to the model. It returns text, not a solution.
3. **Parse and validate.** Text becomes an artifact, or it is refused with a reason.
4. **Loop integration.** Evaluate, accept or reject, keep the best.

Start at the second one, because it is the one that surprises people: what comes
back is a string, and it arrives inside an envelope the validator can find.

In [2]:
pool = load_pool('tsp_pool.txt')
print(pool[0])

CANDIDATE
id: t
representation: permutation
payload:
[0, 1, 4, 4, 2]
END_CANDIDATE


That envelope is the whole contract with the model. `CANDIDATE ... END_CANDIDATE`
delimits the answer, `representation` says which schema the payload follows, and
`payload` carries it. A reply without the envelope is refused at the first layer,
before anyone tries to read a tour out of it.

## The loop, once

This is the code in `search.py`, printed from the module rather than copied here.
There is one copy of Algorithm 1 in this repository and this is it.

In [3]:
import inspect
print(inspect.getsource(build_and_validate))

def build_and_validate(llm, evaluator: Callable[[Any], float],
                       accept: Callable[[Any, float, Any, float], bool],
                       incumbent: Any, spec, budget: int, retries: int,
                       minimize: bool = True) -> Tuple[Any, float, List[tuple]]:
    cur = best = incumbent
    best_score = evaluator(incumbent)
    feedback: List[tuple] = []
    log: List[tuple] = []
    for t in range(budget):
        prompt = spec.render(cur, evaluator(cur), feedback)
        ok, cand, err = False, None, None
        for _ in range(retries + 1):
            text = llm.sample(prompt)
            try:
                cand = spec.parse(text)        # schema + syntax
                spec.feasible(cand)            # domain feasibility
                ok = True
                break
            except ValueError as exc:
                err = str(exc)
                prompt = spec.repair_prompt(prompt, text, err)
        if not ok:
            feedback.append(("inval

## Running it on a problem small enough to hold in your head

The artifact is an integer and lower is better. The spec below is written here
and nowhere else: it is the exercise, not a copy of anything in the modules.

Four functions is all a new problem needs: build the prompt, read the reply,
say whether it is feasible, and say what to send back when it is not.

In [4]:
class IntSpec:
    def render(self, cur, score, feedback):
        return f'[conditioning] incumbent {cur}, score {score}\n[instruction] emit a smaller integer'

    @staticmethod
    def parse(text):
        if 'CANDIDATE' not in text:
            raise ValueError('schema error: no CANDIDATE envelope')
        body = text.split('payload:')[1].split('END_CANDIDATE')[0].strip()
        try:
            return int(body)
        except ValueError:
            raise ValueError('syntax error: payload is not an integer')

    @staticmethod
    def feasible(v):
        if v < 0:
            raise ValueError('feasibility: negative value')
        return True

    def repair_prompt(self, prompt, text, err):
        return prompt + f'\n[repair] {err}'

In [5]:
from llm import envelope

llm = MockLLM([envelope('12', ident='n', representation='integer'),
               envelope('-3', ident='n', representation='integer'),  # refused: not feasible
               envelope('4',  ident='n', representation='integer')])

best, best_score, log = build_and_validate(
    llm, evaluator=lambda v: float(v),
    accept=lambda cand, sc, cur, scur: sc < scur,
    incumbent=20, spec=IntSpec(), budget=3, retries=0, minimize=True)

for step, kind, info in log:
    print(f'step {step}: {kind:8s} {info}')
print('best', best, 'score', best_score)

step 0: accepted 12.0
step 1: invalid  feasibility: negative value
step 2: accepted 4.0
best 4 score 4.0


Three steps, three different outcomes: one candidate accepted, one refused by the
validator before it could be evaluated, one evaluated and rejected by the
acceptance rule. Those are different failures and the log keeps them apart, which
is the point of Algorithm 1 recording a reason and not just a score.

Next: [2. The validator](02_validate_and_repair.ipynb).